In [94]:
import os

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import transforms
from copy import deepcopy

from tqdm.auto import tqdm

import matplotlib.pyplot as plt

import sqlite3
import pandas as pd

import subprocess
from datetime import datetime

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, RESULTS, DB_PATH, MODELS
EMBED_PATH = EMBEDS_DIR/ "cont_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

RESULTS_DIR = RESULTS / EMBED_NAME

MODEL_DIR = MODELS / "anomaly_detection"

In [95]:
os.makedirs(MODEL_DIR, exist_ok=True)

In [96]:
contrastive_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [97]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT path, split
    FROM meta
""", conn)

categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()
types = pd.read_sql_query("SELECT DISTINCT type FROM meta", conn)["type"].to_list()

conn.close()

train_paths = df[df["split"] == "train"]["path"].to_list()
test_paths = df[df["split"] != "train"]["path"].to_list()

In [98]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.12.0+cu130
13.0
True
1


In [99]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

Device: cuda


In [100]:
dino = torch.hub.load(
            "facebookresearch/dinov2",
            "dinov2_vits14"
        )

dino.to(device)
dino.eval()

for p in dino.parameters():
    p.requires_grad = False

Using cache found in /home/ciaran/.cache/torch/hub/facebookresearch_dinov2_main


In [101]:
%load_ext autoreload
%autoreload 2

from src.fine_tune import ModelData, ProjectionHead, BatchSampler, NegativeSampler, train_one_epoch, evaluate

from src.fine_tune.losses.anomaly_head.combined_loss import CombinedLoss

git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True
).strip()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [102]:
projection_dir = MODELS / "category_projection"
if any(projection_dir.iterdir()):
    model_path = max(projection_dir.iterdir(), key=lambda p: p.stat().st_mtime)
    print(model_path)
    
    checkpoint = torch.load(model_path, weights_only=True)
    parameters = checkpoint["model_info"]["parameters"]

    category_head = ProjectionHead(dim=parameters["model_dim"], hidden_dim=parameters["hidden_dim"], norm_type=parameters["model_normaliser"])
    category_head.load_state_dict(checkpoint["model_state_dict"])

    category_head.to(device)
    category_head.eval()
else:
    category_head = None

/home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/models/category_projection/20260722_173228_category_model.pt


In [103]:
inf_models = [dino, category_head]

In [104]:
LOSSES = {
    "cont_loss" : True
}

WEIGHTS = {
  "cont_loss" : 1.0
}

NOTES = [
    "Changed to use a residual connection"
]

In [105]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

SEED = 42

NUM_WORKERS = 8
PIN_MEMORY = device == "cuda"
SAMPLES_PER_CATEGORY = 16

BATCH_SIZE = len(categories) * SAMPLES_PER_CATEGORY

EPOCHS = 20

DIM = 384
HIDDEN_DIM = 768

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MODEL_NORMALISER = "layer"

MODEL_NAME = f"{timestamp}_anomaly_model.pt"

In [106]:
MODEL_INFO = {
    "losses" : LOSSES,
    "weights" : WEIGHTS,
    "notes" : NOTES,
    "git_commit" : git_commit,
    "parameters" : {
        "batch_size" : 256,
        "epochs" : EPOCHS,
        "samples_per_category" : SAMPLES_PER_CATEGORY,

        "model_dim" : DIM,
        "hidden_dim": HIDDEN_DIM,
        "model_normaliser": MODEL_NORMALISER,

        "learning_rate" : LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        
        "num_workers" : NUM_WORKERS,
        "pin_memory" : PIN_MEMORY,

        "random_seed" : SEED
    }
}

In [107]:
category_to_id = {category: idx for idx, category in enumerate(categories)}
types_to_id = {type: idx for idx, type in enumerate(types)}

dataset = ModelData(train_paths, root=ROOT, category_to_id=category_to_id, types_to_id=None, transform=contrastive_transform)
test_set = ModelData(test_paths, root=ROOT, category_to_id=category_to_id, types_to_id=types_to_id, transform=test_transform)

train_split = int(len(dataset) * 0.95)
val_split = len(dataset) - train_split

generator = torch.Generator().manual_seed(SEED)

train_set, val_set = random_split(
    dataset,
    [train_split, val_split],
    generator=generator
)

train_labels = [dataset.category_ids[i] for i in train_set.indices]
val_labels = [dataset.category_ids[i] for i in val_set.indices]

train_sampler = BatchSampler(
    labels=train_labels,
    samples_per_cat=SAMPLES_PER_CATEGORY,
    seed=None
)

val_sampler = BatchSampler(
    labels=val_labels,
    samples_per_cat=SAMPLES_PER_CATEGORY,
    seed=SEED
)

test_labels = test_set.cat_types_ids

negative_sampler = NegativeSampler(
    labels=test_labels,
    samples_per_cat=5,
    types_to_id=types_to_id,
    seed=SEED
)

train_loader = DataLoader(
    train_set,
    batch_sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_set,
    batch_sampler=val_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# Need to change this to remove the negative samples due to data leakage
test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [108]:
neg_indices = negative_sampler.get_negatives()

neg_images = []
neg_labels = []

for idx in neg_indices:
    view1, _, category_id, _ = test_set[idx]

    neg_images.append(view1)
    neg_labels.append(category_id)

neg_images = torch.stack(neg_images).to(device)
neg_labels = torch.tensor(
    neg_labels,
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    neg_embeds = neg_images
    for model in inf_models:
        neg_embeds = model(neg_embeds)

In [109]:
model = ProjectionHead(dim=DIM, hidden_dim=HIDDEN_DIM, norm_type=MODEL_NORMALISER).to(device)

optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = CombinedLoss(enabled=LOSSES, weights=WEIGHTS, negatives=neg_embeds, negative_labels=neg_labels)

In [110]:
results = pd.DataFrame(columns=["epoch", "train_loss", "val_loss", "gap"])

best_val = float("inf")
best_state = None

try:    
    for epoch in tqdm(range(EPOCHS)):
        train_loss = train_one_epoch(model, inf_models, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, inf_models, val_loader, criterion, device)

        if val_loss < best_val:
            best_val = val_loss
            best_state = deepcopy(model.state_dict())
            
        results.loc[len(results)] = {
            "epoch": epoch+1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "gap": abs(val_loss - train_loss)
        }
        
        print(results.tail(1).to_string(
            index=False,
            header=(epoch==0)
        ) + "\n")

finally:
    if best_state is not None:
        checkpoint = {
            "model_state_dict" : best_state,
            "model_info" : MODEL_INFO,
        }
        torch.save(checkpoint, MODEL_DIR / MODEL_NAME)
        print(f"Best Model Saved (val_loss={best_val:.4f})")

  0%|          | 0/20 [00:00<?, ?it/s]

AttributeError: 'EmbeddingBatch' object has no attribute 'negatives'

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(range(EPOCHS), results["train_loss"])
plt.plot(range(EPOCHS), results["val_loss"])

plt.ylabel("Loss")
plt.xlabel("Epoch")

plt.show()